In [ ]:
# Importation des bibliothèques nécessaires
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Configuration pour un meilleur affichage
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

# --- 1. PRÉPARATION DES DONNÉES ---
print("--- 1. PRÉPARATION DES DONNÉES ---")
# Chargement des fichiers
pokemon_df = pd.read_csv('pokemon.csv')
combats_df = pd.read_csv('combats.csv')

print(f"Pokémon DF shape : {pokemon_df.shape}")
print(f"Combats DF shape : {combats_df.shape}")

# 1.1. Correction des valeurs manquantes dans 'pokemon.csv'
# Le Pokémon #62 est 'Primeape'
pokemon_df.loc[pokemon_df['#'] == 62, 'Name'] = 'Primeape'

# Gérer les NaN dans 'Type 2' en les remplaçant par 'None'
pokemon_df['Type 2'] = pokemon_df['Type 2'].fillna('None')

# Vérifier qu'il n'y a plus de valeurs manquantes critiques
print("\nValeurs manquantes après nettoyage :")
print(pokemon_df[['Name', 'Type 2']].isnull().sum())

# 1.2. Calcul du pourcentage de victoire de chaque Pokémon
# Compter le nombre total de combats par Pokémon (comme premier ou second)
all_pokemon_ids = pd.concat([combats_df['First_pokemon'], combats_df['Second_pokemon']]).unique()
win_counts = {pid: 0 for pid in all_pokemon_ids}
total_counts = {pid: 0 for pid in all_pokemon_ids}

for _, row in combats_df.iterrows():
    first, second, winner = row['First_pokemon'], row['Second_pokemon'], row['Winner']
    total_counts[first] += 1
    total_counts[second] += 1
    if winner == first:
        win_counts[first] += 1
    else:
        win_counts[second] += 1

win_rates = {pid: win_counts[pid] / total_counts[pid] if total_counts[pid] > 0 else 0
             for pid in all_pokemon_ids}

# Ajouter le taux de victoire au DataFrame Pokémon
pokemon_df['Win_Rate'] = pokemon_df['#'].map(win_rates)

print(f"\nTaux de victoire calculé pour {len(win_rates)} Pokémon.")
print(pokemon_df[['#', 'Name', 'Win_Rate']].head())

# --- 2. ANALYSE EXPLORATOIRE ET VISUALISATION ---
print("\n--- 2. ANALYSE EXPLORATOIRE ---")

# Sélectionner les colonnes statistiques pertinentes
stat_cols = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Win_Rate']
analysis_df = pokemon_df[stat_cols].copy()

# 2.1. Matrice de corrélation
plt.figure(figsize=(8, 6))
corr_matrix = analysis_df.corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matrice de Corrélation entre Statistiques et Taux de Victoire')
plt.tight_layout()
plt.show()

# 2.2. PairGrid pour les statistiques en fonction du taux de victoire
# Créer des catégories de taux de victoire pour la visualisation
pokemon_df['Win_Rate_Cat'] = pd.cut(pokemon_df['Win_Rate'],
                                    bins=[0, 0.3, 0.5, 0.7, 1.0],
                                    labels=['Faible', 'Moyen', 'Bon', 'Excellent'])

# PairGrid avec les 4 statistiques principales
g = sns.PairGrid(pokemon_df, vars=['HP', 'Attack', 'Defense', 'Speed'],
                 hue='Win_Rate_Cat', palette='viridis')
g.map_diag(sns.histplot, kde=True)
g.map_offdiag(sns.scatterplot, alpha=0.6)
g.add_legend(title='Taux de Victoire')
plt.suptitle('Relations entre Statistiques et Taux de Victoire', y=1.02)
plt.tight_layout()
plt.show()

# 2.3. Top 10 des Pokémon par taux de victoire
print("\nTop 10 des Pokémon avec le meilleur taux de victoire :")
top_10 = pokemon_df.nlargest(10, 'Win_Rate')[['#', 'Name', 'Win_Rate', 'HP', 'Attack', 'Defense', 'Speed']]
print(top_10.to_string(index=False))

print("\nTop 10 des Pokémon avec le taux de victoire le plus faible :")
bottom_10 = pokemon_df.nsmallest(10, 'Win_Rate')[['#', 'Name', 'Win_Rate', 'HP', 'Attack', 'Defense', 'Speed']]
print(bottom_10.to_string(index=False))

# --- 3. MODÉLISATION POUR LA PRÉDICTION ---
print("\n--- 3. MODÉLISATION ---")

# 3.1. Préparation des données pour le ML
# Sélectionner les caractéristiques (features) et la cible (target)
features = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']
# Ajouter des caractéristiques catégorielles encodées (Type 1, Type 2, Legendary)
pokemon_with_dummies = pd.get_dummies(pokemon_df, columns=['Type 1', 'Type 2', 'Legendary'], drop_first=True)
# S'assurer que les colonnes sont alignées et qu'il n'y a pas de valeurs NaN
feature_cols = [col for col in pokemon_with_dummies.columns if col not in ['#', 'Name', 'Win_Rate', 'Win_Rate_Cat']]
X = pokemon_with_dummies[feature_cols].fillna(0)  # Remplacer les éventuels NaN restants
y = pokemon_df['Win_Rate']

# Supprimer les lignes où le taux de victoire est NaN (Pokémon sans combat)
mask = y.notna()
X = X[mask]
y = y[mask]

print(f"Dimensions finales : X={X.shape}, y={y.shape}")

# 3.2. Division entraînement/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3.3. Normalisation des données
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3.4. Entraînement et évaluation des modèles
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, y_pred)
    results[name] = mae
    print(f"{name:20} -> MAE: {mae:.4f}")

# Comparaison des modèles
plt.figure(figsize=(8, 5))
plt.bar(results.keys(), results.values(), color=['blue', 'green', 'red'])
plt.ylabel('Mean Absolute Error (MAE)')
plt.title('Comparaison des Performances des Modèles de Régression')
plt.ylim(0, max(results.values()) * 1.1)
for i, (name, mae) in enumerate(results.items()):
    plt.text(i, mae + 0.001, f'{mae:.4f}', ha='center', va='bottom')
plt.show()

# --- 4. RÉDUCTION DE DIMENSIONNALITÉ PAR ACP ---
print("\n--- 4. ACP (Analyse en Composantes Principales) ---")

# Appliquer l'ACP sur les données standardisées
pca = PCA()
X_pca = pca.fit_transform(X_train_scaled)

# Variance expliquée par composante
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.bar(range(1, len(explained_variance)+1), explained_variance, alpha=0.7)
plt.xlabel('Composante Principale')
plt.ylabel('Variance Expliquée')
plt.title('Variance par Composante')

plt.subplot(1, 2, 2)
plt.plot(range(1, len(cumulative_variance)+1), cumulative_variance, 'bo-', markersize=5)
plt.xlabel('Nombre de Composantes')
plt.ylabel('Variance Cumulée')
plt.title('Variance Cumulée')
plt.axhline(y=0.95, color='r', linestyle='--', label='95%')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Variance expliquée par les 2 premières composantes : {explained_variance[0]:.2%} + {explained_variance[1]:.2%} = {cumulative_variance[1]:.2%}")

# Visualisation des deux premières composantes principales
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_train, cmap='viridis', alpha=0.7, edgecolors='k')
plt.colorbar(scatter, label='Taux de Victoire')
plt.xlabel(f'Première Composante Principale ({explained_variance[0]:.2%} var.)')
plt.ylabel(f'Deuxième Composante Principale ({explained_variance[1]:.2%} var.)')
plt.title('Projection ACP des Pokémon selon leur Taux de Victoire')
plt.tight_layout()
plt.show()

print("\n--- Analyse Terminée ---")